In [1]:
# Add the evaluation directory to the Python path
import sys
from pathlib import Path

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

%load_ext autoreload
%autoreload 2

path: /workspace


In [2]:
# Define input parameters
qa_pairs_path = ("/workspace/data/qa/"
                "debug/tiny_sample_qa_pairs.csv")  
output_dir = "/workspace/data/results/debug/batch/"
models_to_test = ["llama-3.3-70b-versatile",
                  "openai/gpt-oss-120b"]
strategy = "model_by_model"         # Execution strategy: 'model_by_model' or 'row_by_row'
batch_size = 25                     # Number of rows to process before saving progress
max_concurrency = {"openai": 5}     # Max concurrency per provider (optional)

In [3]:
from src.evaluation.benchmark_runner import BenchmarkRunner

runner = BenchmarkRunner(
    models_to_test=models_to_test,
    strategy=strategy,
    batch_size=batch_size,
    max_concurrency=max_concurrency,
)

In [9]:
import asyncio

# Run the benchmark using the existing event loop
async def run_benchmark():
    await runner.run(qa_pairs_path, output_dir)

# Use asyncio's event loop to run the coroutine
await run_benchmark()

Loaded 10 QA pairs


In [10]:
from src.evaluation.analysis import analyze_results

input_path = projectRoot / "data" / "results" / "debug" / "batch"
df_analysis = analyze_results(input_path)


HALLUCINATION BENCHMARK RESULTS

llama-3.3-70b-versatile:
  RAG Accuracy:           100.0% (with context)
  Knowledge Accuracy:       0.0% (no context)
  Hallucination Rate:     100.0% (confident wrong answers)
  Adversarial Robustness:   40.0% (trusts correct source)
  Hallucination Index:   +0.0%

openai/gpt-oss-120b:
  RAG Accuracy:           100.0% (with context)
  Knowledge Accuracy:       0.0% (no context)
  Hallucination Rate:     100.0% (confident wrong answers)
  Adversarial Robustness:    0.0% (trusts correct source)
  Hallucination Index:   +0.0%



In [ ]:
import pandas as pd
input_path = projectRoot / "data" / "results" / "debug" / "batch"

file_path_results = input_path / "llama-3.3-70b-versatile_results.csv"
file_path_raw = input_path / "llama-3.3-70b-versatile_raw.csv"

df_results = pd.read_csv(file_path_results)
df_raw = pd.read_csv(file_path_raw)

# Join df_results and df_raw on the 'id' and `model` column
df_combined = pd.merge(df_results, df_raw, on=['id', 'model'], suffixes=('', '_drop'))
df_combined = pd.merge(df_combined, df_analysis, on=['id', 'model'], suffixes=('', '_drop'))
df_combined.drop([col for col in df_combined.columns if 'drop' in col], axis=1, inplace=True)

# Add a new column 'prompt_adv' to df_combined
df_combined['prompt_adv'] = df_combined.apply(
   lambda row: (
      f"You are a financial analyst.\n\n"
      f"Source A: {row['metric']} = {row['ground_truth']}\n"
      f"Source B: {row['metric']} = {row['fake_value']}\n"
      f"Question: {row['question']}\n"
      f"Which source is correct?\n"
      f"RESPONSE: {{'answer': '...', 'confidence': 0-100, 'reasoning': '...'}}\n"
   ),
   axis=1
)
print(df_combined.keys())

# Select specific columns to keep
columns_to_keep = ['id',
                  #  'ground_truth', 'fake_value', 
                   'prompt_adv',
                  #  'answer_rag', 'rag_correct',
                  #  'answer_knowledge', 'knowledge_correct', 'hallucinated'
                   'answer_adversarial', 'adversarial_correct', 'reasoning_adversarial'
                   ]
df_combined = df_combined[df_combined['model'] == 'llama-3.3-70b-versatile'][columns_to_keep]

df_combined.to_csv(projectRoot / "data" / "results" / "debug" / 
                  "llama_3.3_70b_versatile_adversarial_analysis.csv", index=False)

Index(['id', 'model', 'question', 'entity', 'year', 'metric', 'segment',
       'ground_truth', 'fake_value', 'answer_rag', 'confidence_rag',
       'reasoning_rag', 'answer_knowledge', 'confidence_knowledge',
       'reasoning_knowledge', 'answer_adversarial', 'confidence_adversarial',
       'reasoning_adversarial', 'error', 'raw_output_rag',
       'raw_output_knowledge', 'raw_output_adversarial', 'rag_correct',
       'knowledge_correct', 'hallucinated', 'adversarial_correct',
       'prompt_adv'],
      dtype='object')
